# Series de Taylor

In [1]:
# Librerías utilizadas en este capítulo
import numpy as np
import math                                  # para el factorial
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML

plt.rcParams.update({'font.size': 10, 'figure.dpi': 200})

## Introducción

Un péndulo simple de largo $L$ obedece la ecuación de movimiento

\begin{equation*}
\frac{d^2\theta}{dt^2} + \frac{g}{L}\sin\theta = 0
\end{equation*}

<img src="./images/pendulo.png" width="330" align= center>

Esta ecuación no tiene solución analítica. En todos los cursos de física se resuelve reemplazando $\sin\theta$ por $\theta$, lo que la convierte en la ecuación del oscilador armónico, de periodo $T_0 = 2\pi\sqrt{L/g}$.

> ¿Hasta qué ángulo vale ese reemplazo? La pregunta no se responde mirando el gráfico de $\sin\theta$. Necesitamos una fórmula que entregue el error, y esa fórmula es la serie de Taylor.

En la [Unidad 4](../04-Interpolacion/04-Interpolacion.ipynb) aproximamos una función por un polinomio que pasa por $n+1$ datos. Acá también aproximamos por un polinomio, pero con otra información de entrada: en vez de $n+1$ puntos usamos las primeras $n+1$ derivadas en un solo punto.

> Es el mismo problema canónico del curso, aproximar por un polinomio, alimentado con un dato distinto. La interpolación necesita una tabla; Taylor necesita un punto y sus derivadas.

## Expansión de funciones en series de Taylor

La expansión en series de Taylor es una forma alternativa de representar una función mediante una serie infinita de polinomios alrededor de un punto $a$:

\begin{equation*}
f(x) = \sum_{n = 0}^{\infty} \frac{f^{(n)}(a)}{n!}(x-a)^n
\end{equation*}

donde $f^{(n)}$ es la $n$-ésima derivada de $f$, con $f^{(0)} = f$. Al punto $a$ lo llamamos **punto de expansión**.

Derivemos la expansión de $f(x) = 5x^2 + 3x + 5$ alrededor de $a = 0$. Las derivadas son $f'(x) = 10x+3$, $f''(x) = 10$ y $f'''(x) = 0$, de modo que:

\begin{align*}
f(x) &= \frac{5}{0!}x^0 + \frac{3}{1!}x^1 + \frac{10}{2!}x^2 + 0 + \cdots \\
     &= 5x^2 + 3x + 5
\end{align*}

> Si repetimos el cálculo en $a = 2$ obtenemos $31 + 23(x-2) + 5(x-2)^2$, que al expandir vuelve a ser $5x^2 + 3x + 5$. **La expansión de Taylor de un polinomio es el mismo polinomio**, cualquiera sea el punto de expansión.

## Aproximación de funciones no polinomiales

Para funciones que no son polinomios, la serie de Taylor tiene infinitos términos.

Por ejemplo, la expansión de $f(x) = \sin(x)$ alrededor de $a=0$:

\begin{align*}
f(x) &= \frac{\sin(0)}{0!}x^0 + \frac{\cos(0)}{1!}x^1 + \frac{-\sin(0)}{2!}x^2 + \frac{-\cos(0)}{3!}x^3 + \frac{\sin(0)}{4!}x^4 + \frac{\cos(0)}{5!}x^5 + \cdots \\
     &= \frac{x^1}{1!} - \frac{x^3}{3!} + \frac{x^5}{5!} - \cdots
\end{align*}

Analizando el patrón de la serie, tenemos que:

\begin{equation*}
\sin(x) = \sum_{n = 0}^{\infty} \frac{(-1)^n x^{2n+1}}{(2n+1)!}
\end{equation*}

La ventaja de esta representación aparece al evaluar la función en un computador: un polinomio solo requiere sumar, restar y multiplicar, que es todo lo que un procesador sabe hacer.

Aunque la serie tiene infinitos términos, obtenemos una buena aproximación considerando solo los primeros. Es decir, **truncamos la serie de Taylor**:

\begin{equation*}
\sin(x) \approx \sum_{n = 0}^{N} \frac{(-1)^n x^{2n+1}}{(2n+1)!}
\end{equation*}

> Decimos que la aproximación de una función es de **orden $N$** cuando la serie considera hasta el término $N$.

Analicemos cómo mejora la aproximación de $\sin(x)$ al aumentar el orden. Luego, revisemos como funciona la aproximación mediante series de Taylor para las funciones: $e^x$, $\ln(1+x)$ y $\frac{1}{1-x}$.

In [4]:
HTML(Path('interactive/A1_taylor_bajo_la_lupa.html').read_text(encoding='utf-8'))

x,1.000
f(x),–
pN(x),–
error |f − pN|,–
cota |RN|,–


En la animación, vemos que el error máximo en $[-\pi,\pi]$ baja de 3.1 con la recta a $6{,}9\times10^{-3}$ con el orden 9. Cerca de $x=0$ incluso la recta sirve, y la diferencia se concentra en los bordes del intervalo.

**Subir el orden no siempre ayuda.** En $1/(1-x)$ evaluada en $x = 1.2$ la serie diverge: con $N=5$ entrega 9.9, con $N=20$ entrega 225, y el valor correcto es $-5$. La serie de Taylor solo converge dentro de un radio alrededor de $a$.

>**Nota.** Los métodos numéricos usan casi siempre la aproximación de primer orden, llamada **aproximación lineal**, porque es suficientemente buena en valores cercanos al punto de expansión. En otras palabras, *una función suave siempre se ve como una recta si la miramos suficientemente cerca.*

## El resto de Taylor

Hasta aquí dijimos "a mayor orden, mejor la aproximación". La serie de Taylor permite decir **cuánto** mejor.

Truncando en el orden $N$, la igualdad exacta es:

\begin{equation*}
f(x) = \underbrace{\sum_{n=0}^{N}\frac{f^{(n)}(a)}{n!}(x-a)^n}_{p_N(x)} \; + \; \underbrace{\frac{f^{(N+1)}(\xi)}{(N+1)!}(x-a)^{N+1}}_{R_N(x)}
\end{equation*}

donde $\xi$ es algún punto entre $a$ y $x$.

> El **resto** $R_N$ no es una estimación, es la igualdad exacta. Lo que no conocemos es $\xi$. Por eso en la práctica lo usamos como cota superior: reemplazamos $f^{(N+1)}(\xi)$ por el valor máximo que esa derivada alcanza en el intervalo.

Volvamos al péndulo. La aproximación $\sin\theta \approx \theta$ es de orden 1, así que el resto es

\begin{equation*}
|R_2| = \left|\frac{-\cos\xi}{3!}\theta^3\right| \le \frac{\theta^3}{6}
\end{equation*}

In [14]:
theta_deg = np.array([5, 10, 14, 15, 30, 45])
theta     = np.radians(theta_deg)

error_abs = np.abs(np.sin(theta) - theta)      # error real de la aproximación
cota      = theta**3/math.factorial(3)         # cota del resto de Taylor
error_rel = error_abs/np.abs(np.sin(theta))    # error relativo

for td, ea, c, er in zip(theta_deg, cota, error_abs, error_rel):
    print(f'theta = {td:2.0f}° | R2 = {c:.2e} | error abs. = {ea:.2e} | error rel. = {er*100:5.2f} %')

theta =  5° | R2 = 1.11e-04 | error abs. = 1.11e-04 | error rel. =  0.13 %
theta = 10° | R2 = 8.85e-04 | error abs. = 8.86e-04 | error rel. =  0.51 %
theta = 14° | R2 = 2.42e-03 | error abs. = 2.43e-03 | error rel. =  1.00 %
theta = 15° | R2 = 2.98e-03 | error abs. = 2.99e-03 | error rel. =  1.15 %
theta = 30° | R2 = 2.36e-02 | error abs. = 2.39e-02 | error rel. =  4.72 %
theta = 45° | R2 = 7.83e-02 | error abs. = 8.07e-02 | error rel. = 11.07 %


La cota predice el error real con menos de 4 % de margen en todo el rango. El reemplazo $\sin\theta\approx\theta$ cuesta menos de 1 % hasta los 14°, y 4,7 % a los 30°. Eso es lo que significa "oscilaciones pequeñas".

## La forma en paso $h$

Hasta aquí escribimos la serie en $(x-a)$. En el resto del curso el punto de expansión es un nodo $x_i$ y la distancia es el **paso** $h$.

\begin{equation*}
f(x_i + h) = f(x_i) + f'(x_i)h + \frac{f''(x_i)}{2!}h^2 + \cdots + \frac{f^{(N)}(x_i)}{N!}h^N + O(h^{N+1})
\end{equation*}

Es la misma ecuación, con $a = x_i$ y $x = x_i + h$. La notación $O(h^{N+1})$ resume el resto: si $h$ se reduce a la mitad, un término $O(h^2)$ se reduce cuatro veces.

> Esta es la forma en que la serie de Taylor aparece en el resto del curso. El **orden** de un método numérico es la potencia de $h$ del primer término que se descarta.

## Errores de truncamiento

En la **Unidad 1** vimos los errores de redondeo, inducidos por la capacidad limitada del computador para almacenar decimales. Al aproximar una función por una serie de orden $N$ aparece un segundo tipo: el **error de truncamiento**.

Por ejemplo, la expansión de $e^x$ alrededor de $a=0$ es:

\begin{equation*}
e^x = \sum_{n=0}^{\infty}\frac{x^n}{n!} = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots
\end{equation*}

Analicemos el error de truncamiento de $e^2$ en función del orden $N$.

In [6]:
x = 2.0        # evaluamos exp(2)
N = 9          # orden máximo

exp_approx = 0
for n in range(N + 1):
    exp_approx += x**n/math.factorial(n)
    print(f'orden {n}: aprox = {exp_approx:.6f}, error = {np.abs(exp_approx - np.exp(x)):.3e}')

orden 0: aprox = 1.000000, error = 6.389e+00
orden 1: aprox = 3.000000, error = 4.389e+00
orden 2: aprox = 5.000000, error = 2.389e+00
orden 3: aprox = 6.333333, error = 1.056e+00
orden 4: aprox = 7.000000, error = 3.891e-01
orden 5: aprox = 7.266667, error = 1.224e-01
orden 6: aprox = 7.355556, error = 3.350e-02
orden 7: aprox = 7.380952, error = 8.104e-03
orden 8: aprox = 7.387302, error = 1.755e-03
orden 9: aprox = 7.388713, error = 3.436e-04


> Cada término nuevo reduce el error, pero cada vez menos: de $6.4$ con $N=0$ a $3.4\times10^{-4}$ con $N=9$. Ese es el comportamiento típico del truncamiento, y el resto de Taylor explica su velocidad.

## Errores de redondeo

Una serie de Taylor se calcula sumando términos sucesivos, y cada suma arrastra error de máquina. El efecto es más evidente cuando el valor exacto es pequeño.

Consideremos la aproximación de $e^{-30}$ usando 200 términos de la serie.

In [7]:
x = -30        # entero: x**i/math.factorial(i) es división exacta entre enteros grandes

exp_approx = 0
for i in range(200):
    exp_approx += x**i/math.factorial(i)

print(f'Aproximación de orden {i}: {exp_approx:.5e}')
print(f'Valor exacto:              {np.exp(x):.5e}')

Aproximación de orden 199: -8.55302e-05
Valor exacto:              9.35762e-14


A pesar de haber incluido 200 términos, la estimación es errónea en todas sus cifras. Veamos por qué.

In [8]:
terminos = [x**i/math.factorial(i) for i in range(200)]
i_max    = np.argmax(np.abs(terminos))

print(f'Término más grande de la serie: {terminos[i_max]:.3e} (i = {i_max})')
print(f'Su última cifra significativa:  {np.finfo(float).eps*abs(terminos[i_max]):.3e}')
print(f'Valor que estamos buscando:     {np.exp(x):.3e}')

Término más grande de la serie: -7.762e+11 (i = 29)
Su última cifra significativa:  1.724e-04
Valor que estamos buscando:     9.358e-14


> El problema no es la serie, es la **cancelación**. Sumamos términos del orden de $10^{11}$ para obtener $10^{-13}$, y el error de máquina de esos términos es $10^{-4}$: nueve órdenes de magnitud más grande que la respuesta.

La solución es reordenar el cálculo para que no haya restas entre números grandes. Como $e^{-30} = 1/e^{30}$, y la serie de $e^{30}$ tiene todos sus términos positivos:

In [9]:
serie_positiva = sum(30**i/math.factorial(i) for i in range(200))   # serie de exp(+30)

print(f'1/serie(e^30) = {1/serie_positiva:.6e}')
print(f'Valor exacto  = {np.exp(-30):.6e}')

1/serie(e^30) = 9.357623e-14
Valor exacto  = 9.357623e-14


> La misma serie, con el mismo número de términos, ahora acierta en todas sus cifras. **Cómo se ordena un cálculo importa tanto como la fórmula.**

## El error total y el paso óptimo

Los dos errores se oponen. El de truncamiento baja al usar más términos o un paso más chico; el de redondeo sube, porque cada operación adicional acumula error de máquina.

Veámoslo en la aproximación más simple que sale de truncar la serie en primer orden, la derivada numérica:

\begin{equation*}
f'(x) \approx \frac{f(x+h) - f(x)}{h}, \qquad \text{con error de truncamiento } O(h)
\end{equation*}

> El numerador resta dos números casi iguales y luego divide por un $h$ pequeño, de modo que el error de redondeo crece como $\varepsilon_\mathrm{maq}/h$. El error total es la suma de una recta que baja y otra que sube.

En la siguiente animación, movamos $h$ y observemos dónde el error total deja de bajar.

In [10]:
HTML(Path('interactive/A2_h_optimo.html').read_text(encoding='utf-8'))

f'(x) numérica,–
f'(x) exacta,–
error total,–
cifras correctas,–


> Para $f=\sin$ evaluada en $x=1$, el error mínimo es $3\times10^{-9}$ en $h\approx 10^{-8}$, y con $h=10^{-16}$ sube hasta 0.54. La teoría predice $h_\mathrm{opt}\approx\sqrt{\varepsilon_\mathrm{maq}} = 1{,}5\times10^{-8}$: bajar el paso más allá de ese punto empeora el resultado.

> Esta curva en V reaparece en la **Unidad 8**, al comparar fórmulas de derivación numérica de distinto orden. Cada orden mueve el mínimo a otro lugar.

Antes de confiar en una aproximación de Taylor conviene revisar siempre:

1. ¿El punto de evaluación está cerca del punto de expansión? El error crece como $|x-a|^{N+1}$.
2. ¿La función tiene sus derivadas en todo el intervalo? Sin eso no existe el resto.
3. ¿El cálculo resta números grandes y parecidos? Ahí el redondeo manda sobre el truncamiento.
4. ¿Bajar el paso siguió mejorando el resultado? Si dejó de mejorar, ya pasamos el $h$ óptimo.

## Referencias

- Chapra S. **Chapter 4: Roundoff and Truncation Errors** en *Applied Numerical Methods with MATLAB for Engineers*, 3rd Ed., McGraw Hill.
  - §4.2 errores de redondeo · §4.3 errores de truncamiento y el resto · §4.4 error total y el paso óptimo

- Press W., Teukolsky S., Vetterling W., Flannery B. **Chapter 5: Evaluation of Functions** en *Numerical Recipes: The Art of Scientific Computing*, 3rd Ed., Cambridge University Press, 2007.
  - §1.1 error, exactitud y estabilidad · §5.1 evaluación de polinomios · §5.3 series y su convergencia · §5.7 derivadas numéricas y el paso óptimo

- Kong Q., Siauw T., Bayen A. M. **Chapter 18: Taylor Series** en *[Python Programming and Numerical Methods – A Guide for Engineers and Scientists](https://pythonnumericalmethods.berkeley.edu/notebooks/chapter18.00-Series.html)*, 1st Ed., Academic Press, 2021